# Lecture 6: ODE solvers

## 1) Motivation

**Goal:** Solve the time-dependent Schrödinger Equation (SE):

$$\dot{\vec{\psi}} = -i H \vec{\psi}, \quad \vec{\psi}(t=0) = \vec{\psi}_0$$

Formally, for a time-independent Hamiltonian $H$:

$$\vec{\psi}(t) = e^{-iHt} \vec{\psi}_0$$

---

### So far we used exact-diagonalization:

$$H = V D V^\dagger$$

* Where eigenvectors of $H$ are the columns of $V$
* Eigenvalues of $H$ are on the diagonal of $D$

$$\Rightarrow \vec{\psi}(t) = V e^{-iDt} V^\dagger \vec{\psi}_0$$

Where $e^{-iDt}$ is the diagonal matrix:

$$\begin{pmatrix}
e^{-iE_1 t} & & 0 \\
& \ddots & \\
0 & & e^{-iE_d t}
\end{pmatrix}$$

---

### Problems:
* $V$ is **dense** $\Rightarrow V \vec{\psi}_0$ scales as $d^2$ 
  * $+$ memory limitations for large $d$
* For time-dependent problems $H(t)$, this approach is not feasible.

---

### Solution: Discretize time into steps $\Delta t$

$$\vec{\psi}(t) = \left(e^{-iH\Delta t}\right)^n \vec{\psi}_0 \quad (t = n \Delta t)$$

$$\Rightarrow \text{How to obtain } \vec{\psi}(t+\Delta t) = e^{-iH\Delta t} \vec{\psi}(t) \text{?}$$

* What is the error?
* Is it stable, norm conserving, ...?

## 2) Taylor series propagators

> **Notation Note:**
> * $\Delta t \longleftrightarrow dt$
> * $\vec{\psi} \longleftrightarrow |\psi\rangle$

### - Euler method:

$$|\psi(t+dt)\rangle = e^{-iHdt}|\psi(t)\rangle = (1 - iHdt)|\psi(t)\rangle + \mathcal{O}(dt^2)$$

* **Error:** $\sim dt^2$ per time step (global error $\sim dt$ for a fixed $t_{\text{end}}$).
* **Stability:** Calculate the norm:

$$\langle\psi(t+dt)|\psi(t+dt)\rangle = \langle\psi(t)|(1 + idtH)(1 - idtH)|\psi(t)\rangle$$

$$= 1 + dt^2 \langle\psi(t)|H^2|\psi(t)\rangle$$

$$\rightarrow \text{\textbf{Norm grows!}}$$

After $n_t$ steps:

$$\|\psi(t)\| = \langle\psi_0|(1 + dt^2 H^2)^{n_t}|\psi_0\rangle \sim e^{t \, dt \, \lambda_{\text{max}}^2}$$

*(where $\lambda_{\text{max}}$ is the largest eigenvalue of $H$)*

$$\Rightarrow \text{\textbf{norm explodes exponentially}} \Rightarrow \text{\textbf{unstable}}$$

$$\Rightarrow \text{need tiny time steps } dt \ll \frac{1}{\lambda_{\text{max}}}$$

---

### - $n$-th order:

$$|\psi(t+dt)\rangle = \left( \sum_{j=0}^{n} \frac{(-idt)^j}{j!} H^j \right) |\psi(t)\rangle + \mathcal{O}(dt^{n+1})$$

* Error per step $\sim dt^{n+1} \rightarrow$ larger steps possible.
* But **still unstable**, not norm-conserving.

---

> **Note:** For a linear ODE $\dot{\psi} = H\psi$, Runge-Kutta methods reduce to the Taylor expansion method.

---

### General stability criterion:

$$|\psi(t+\Delta t)\rangle = A|\psi(t)\rangle : \text{ if } \|A^\dagger A\| > 1 \rightarrow \text{\textbf{unstable}}$$

*(where $\|\cdot\|$ is the 2-norm, i.e., the absolute value of the largest eigenvalue)*

### Stiffness:

$$\left| \frac{\lambda_{\text{max}}}{\lambda_{\text{min}}} \right| \gg 1 \quad \text{problem is "stiff"}$$

* **Physically:** Need small steps to resolve dynamics on the time scale $1/\lambda_{\text{max}}$. Need to integrate to long times to observe dynamics on the time scale $1/\lambda_{\text{min}}$.
  $$\Rightarrow \text{\textbf{Many steps needed.}}$$

---

### Advantage of explicit methods:
* Only $H|\psi\rangle$-multiplication needed.
  $$\rightarrow \text{\textbf{efficient if } } H \text{ \textbf{is sparse!}} \sim d$$

## 3) Implicit methods: Crank-Nicolson

**Idea:**

$$e^{+i \frac{dt}{2} H} |\psi(t+dt)\rangle = e^{-i \frac{dt}{2} H} |\psi(t)\rangle$$

$$\underbrace{\left(1 + i \frac{dt}{2} H\right)}_{A} \underbrace{|\psi(t+dt)\rangle}_{x} = \underbrace{\left(1 - i \frac{dt}{2} H\right) |\psi(t)\rangle}_{b}$$

* **"Implicit"** $\rightarrow$ To obtain $|\psi(t+dt)\rangle$ we have to solve a linear system $Ax = b$ 
  $$\rightarrow \text{more costly than explicit methods but \textbf{more stable.}}$$
  $$\Rightarrow \text{\textbf{Can do larger time steps!}}$$

---

$$|\psi(t+dt)\rangle = \underbrace{\frac{1 - i \frac{dt}{2} H}{1 + i \frac{dt}{2} H}}_{A} |\psi(t)\rangle$$

$$A^\dagger A = \mathbb{1} \Rightarrow \text{\textbf{norm-conserving}}, \, A \text{ is unitary}$$

$$\text{error } \sim dt^3 \text{ per step}$$

## 4) Krylov subspace propagator

Recall $n$-th order Taylor series:

$$|\psi(t+dt)\rangle = \left( 1 - idtH - \frac{1}{2} dt^2 H^2 \pm \dots \right) |\psi(t)\rangle$$

$$= \text{superpos. of vectors } (-iH)^k |\psi(t)\rangle \quad (k = 0 \dots n)$$
$$\text{with decreasing weight } \frac{dt^k}{k!}$$

$$\rightarrow |\psi(t+dt)\rangle \text{ lies in Krylov space } \mathcal{K}_{n+1}(-iH, |\psi(t)\rangle)$$
$$\text{spanned by vectors } (-iH)^k |\psi(t)\rangle \text{ up to an error } \sim dt^{n+1}$$

---

### Idea: Evolve unitarily in Krylov space.

Recall: $\{q_0, \dots, q_n\}$ are orthonormalized Krylov vectors.

$$Q = \begin{pmatrix} q_0 \dots q_n \end{pmatrix} \updownarrow d \quad \text{rectangular matrix} \quad (\leftrightarrow n)$$

$$h = Q^\dagger H Q \quad \text{Hamiltonian within Krylov space}$$

---

### "Algorithm":
* Project $|\psi(t)\rangle$ into Krylov space
* Evolve with $e^{-ihdt}$. Can be done exactly as $h$ ($n \times n$ tri-diagonal matrix is easily diagonalized)
* Express $|\psi(t+dt)\rangle$ in original basis

---

### Mathematically:

$$|\psi(t+dt)\rangle = Q e^{-ihdt} Q^\dagger |\psi(t)\rangle$$

$$= Q e^{-ihdt} e_1$$

$$\text{(recall that } |\psi(t)\rangle \text{ is the first Krylov-vector } q_0)$$

$$\rightarrow Q^\dagger |\psi(t)\rangle = \begin{pmatrix} 1 \\ 0 \\ \vdots \\ 0 \end{pmatrix} \updownarrow n \equiv e_1 \text{ )}$$

This scheme is obviously unitary (norm-preserving) since $Q e^{-ihdt} Q^\dagger$ is a unitary operator.

**Claim:** This scheme is an order $dt^{n+1}$ propagation just as the $n$-th order Taylor series propagator.

$$Q e^{-ihdt} Q^\dagger = e^{-i Q h Q^\dagger dt} \equiv e^{-i \tilde{H} dt} \overset{!}{=} e^{-i H dt} + \mathcal{O}(dt^{n+1})$$

To prove the last equality, we need to show that:

$$H^k \psi = \tilde{H}^k \psi \quad \text{for } k \le n$$

---

### Proof by induction:

Assuming $H^{k-1} \psi = \tilde{H}^{k-1} \psi$, we have:

$$H^k \psi = \underbrace{Q Q^\dagger H^k \psi}_{H^k \psi \text{ is inside of } \mathcal{K}_{n+1}} = Q Q^\dagger H \underbrace{\tilde{H}^{k-1} \psi}_{\text{ind. assumption}} = Q Q^\dagger H Q h^{k-1} Q^\dagger \psi$$

$$= Q h^k Q^\dagger \psi = \tilde{H}^k \psi$$

$$\begin{pmatrix} q_0 \dots q_n \end{pmatrix} \begin{pmatrix} q_0^\dagger \\ \vdots \\ q_n^\dagger \end{pmatrix} \underbrace{\sum_{k=0}^n c_k q_k}_{\parallel \atop H^k \psi} = \begin{pmatrix} q_0 \dots q_n \end{pmatrix} \begin{pmatrix} c_0 \\ \vdots \\ c_n \end{pmatrix} = \sum c_k q_k = H^k \psi$$

---

### Advantages:
* Error strictly smaller than for $n$-th order Taylor.
* Norm-preserving.
* Only $H\psi$-multiplications (no matrix inversion).

Typically $n \sim 20$ is optimal $\Rightarrow$ large time steps possible.

**"Gold standard"** for propagating SE with sparse $H$.
* Error estimates available.

## 5) Adaptive step-size

**Goal:** In each time step, make the step size $dt$ small enough to meet a given accuracy $\varepsilon$.

* If an error estimate is available (upper bound on error), check and reduce step size by some factor until $\Delta_{\text{err}} \le \varepsilon$. If $\Delta_{\text{err}} < \varepsilon$, keep or increase $dt$.

* **Error estimation by half step:**
  * Calculate $\psi(t+dt)$ with step $dt$
  * Calculate $\tilde{\psi}(t+dt)$ by evolving in 2 half steps $\frac{dt}{2}$
  
$$\rightarrow \text{error estimate } \Delta_{\text{err}} = \|\psi(t+dt) - \tilde{\psi}(t+dt)\|$$